In [47]:
import os
import sys
from pathlib import Path

# find the main project root folder automatically
current_dir = Path().resolve()
project_root = current_dir
while project_root != project_root.parent:
    if (project_root / "src").exists():
        break
    project_root = project_root.parent

# add src to sys.path so your imports still work
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# set the working directory to the 'src' folder
os.chdir(project_root / "src")

In [85]:
# This LSTMRNN class was taken from Notebook ‘5.1 Recurrent Neural Networks’ (original class name is "MyRecurrentNet"), which was provided 
# as part of the course materials for Week 5. Modifications have been made to complete the implementation.

import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMRNN(nn.Module):
    def __init__(self, n_in, n_hid,nb_layers,bias,families):
        super(LSTMRNN, self).__init__()

        # Context window
        # n_in = train_data.max_length
        # self.context_window = context_window
        # window_emb = self.context_window * embedding_dim
        # self.embedding = nn.Embedding(num_embeddings=n_in, embedding_dim=embedding_dim, padding_idx=IGNORE_INDEX)

        # Recurrent layer
        self.lstm = nn.LSTM(input_size=n_in, hidden_size=n_hid, num_layers=nb_layers, bias=bias, batch_first=True)
        
        # Output layer
        self.l_out = nn.Linear(in_features=n_hid, out_features=families, bias=False)
        
    def forward(self, x):
        # RNN does the embedding

        # RNN returns output and last hidden state
        x, (h, c) = self.lstm(x)
        
        # Flatten output for feed-forward layer
        x = x.view(-1, self.lstm.hidden_size)
        
        # Output layer
        x = self.l_out(x)
        
        return x

In [66]:
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(train_data[3])

(tensor([ 1, 13, 14, 11, 19,  6, 12, 10, 12, 17, 10,  3,  6,  6, 19,  8,  7, 18,
        17, 11, 14, 20,  6, 12,  8, 12, 14,  3,  3, 10, 14, 12, 10, 19, 11,  3,
        12, 19,  6,  6,  6, 11, 20, 16, 10, 20,  8,  7,  8,  3,  7,  6, 19, 11,
        20, 17,  3,  6, 17, 13,  8, 17, 14, 15, 11, 19,  8,  6,  5, 10, 15, 10,
        15,  3, 19, 11, 20, 15, 20,  7, 11,  3,  8, 11,  3, 12, 11,  5,  3, 20,
         5, 16,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0]), tensor([13, 14, 11, 19,  6, 12, 10, 12, 17, 10,  3,  6,  6, 19,  8,  7, 18, 17,
        11, 14, 20,  6, 12,  8, 12, 14,  3,  3, 10, 14, 12, 10, 19, 11,  3, 12,
        19,  6,  6,  6, 11, 20, 16, 10, 20,  8,  7,  8,  3,  7,  6, 19, 11, 20,
        17,  3,  6, 17, 13,  8, 17, 14, 15, 11, 19,  8,  6,  5, 10, 15, 10, 15,
         3, 19, 11, 20, 15, 20,  7, 11,  3,  8, 11,  3, 12, 11,  5,  3, 20,  5,
        16,  2, -

In [93]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

# These methods were taken from Notebook ‘5.1 Recurrent Neural Networks’, which was provided as part of the course materials for Week 5. 

import numpy as np
import torch

def one_hot_encode(idx, vocab_size):
    """
    One-hot encodes a single word given its index and the size of the vocabulary.
    
    Args:
     `idx`: the index of the given word
     `vocab_size`: the size of the vocabulary
    
    Returns a 1-D numpy array of length `vocab_size`.
    """
    # Initialize the encoded array
    one_hot = np.array([np.zeros(vocab_size)])
    
    # Set the appropriate element to one
    one_hot[0][idx] = 1.0
    return one_hot

# This method from the referenced Notebook has been updated to align with the vocabulary mapping from ProteinDataset.
def one_hot_encode_sequence(sequence, vocab_size):
    """
    One-hot encodes a sequence of words given a fixed vocabulary size.
    
    Args:
     `sentence`: a list of words to encode
     `vocab_size`: the size of the vocabulary
     
    Returns a 3-D torch.Tensor of shape (num words, vocab size, 1).
    """
    # Encode each word in the sentence
    encoding = np.array([one_hot_encode(int(idx), vocab_size) for idx in sequence])

    # Reshape encoding s.t. it has shape (num words, vocab size, 1)
    encoding = encoding.reshape(encoding.shape[0], encoding.shape[2], 1)
    return torch.tensor(encoding)

N_IN = 50
N_HID = 128
NUM_LAYERS = 2
NUM_EPOCHS = 200
BOS = "<BOS>"
PAD = "<PAD>"
EOS = "<EOS>"
BOS_IDX = 1
PAD_IDX = 0
EOS_IDX = 2
CSV_FILENAME = "lstm_samples.csv"
MODEL_NB = 4
MODEL_NAME = "lstm_train_model.pth"
# early stopping parameters
MAX_EPOCHS_WITHOUT_IMPROVEMENT = 10
MIN_IMPROVEMENT = 0.001

In [ ]:
import torch.optim as optim
from torch import nn
import torch
import matplotlib.pyplot as plt
import numpy as np
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens
import time
from datetime import datetime

In [88]:
# This code was taken from Notebook ‘5.1 Recurrent Neural Networks’, which was provided as part of the course materials for Week 5. 
# Modifications have been made to complete the implementation. The file also contains code from Notebook '4.1 CNN Introduction'.ipynb,
# which was provided as part of the course materials for Week 4.

# Ensure script is run on GPU rather than CPU
device = torch.device("cuda")

# Initialize a new network
net = LSTMRNN(n_in= train_data.vocab_size, n_hid=N_HID, nb_layers=1, bias=True, families=3)#.to(device)

# Define a loss function and optimizer for this problem
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX) # using `ignore_index` to make the padding free of loss as recommended in the introduction.ipynb notebook
optimizer = optim.Adam(net.parameters(), lr=1e-3)

In [91]:
families = train_data.fam_to_idx
idx_to_family = {idx: token for token, idx in families.items()}

# Track loss
training_loss, validation_loss = [], []

# For each epoch
for i in range(NUM_EPOCHS):
    epoch_start = time.time()
    
    # Track loss
    epoch_training_loss = 0
    epoch_validation_loss = 0
    
    net.eval()
        
    # For each sentence in validation set
    with torch.no_grad():
        for inputs, targets, labels, lengths in val_data:
            
            # One-hot encode input and target sequence
            inputs_one_hot = one_hot_encode_sequence(inputs, train_data.vocab_size)
            
            # Convert input to tensor
            inputs_one_hot = torch.as_tensor(inputs_one_hot, dtype=torch.float32).permute(2,0,1)
            
            # Convert target to tensor
            targets_ = torch.as_tensor(labels, dtype=torch.long)
            
            # Forward pass
            outputs = net(inputs_one_hot)
            print(outputs.shape)
            
            # Compute loss
            loss = criterion(outputs, targets_)
            
            # Update loss
            epoch_validation_loss += loss.item()
    
    net.train()
    
    # For each sentence in training set
    for inputs, targets, labels, lengths in train_data:

        # One-hot encode input and target sequence
        inputs_one_hot = one_hot_encode_sequence(inputs, train_data.vocab_size)
        
        # Convert input to tensor
        inputs_one_hot = torch.as_tensor(inputs_one_hot, dtype=torch.float32).permute(2,0,1)
        
        # Convert target to tensor
        targets_ = torch.as_tensor(targets, dtype=torch.long)
        
        # Forward pass
        outputs = net(inputs_one_hot)
        
        # Compute loss
        loss = criterion(outputs, targets_)
        
        # Clean up gradients from the model
        optimizer.zero_grad()

        # Compute gradients based on the loss from the current batch (backpropagation).
        loss.backward()
        
        # Take one optimizer step using the gradients computed in the previous step.
        optimizer.step()
        
        # Update loss
        epoch_training_loss += loss.item()

    # Save loss for plot
    training_loss.append(epoch_training_loss/len(train_data))
    validation_loss.append(epoch_validation_loss/len(val_data))

    # Save model
    torch.save({"model": net.state_dict(), "criterion": criterion.state_dict(), "optimizer": optimizer.state_dict(), "train_loss": training_loss, "val_loss": validation_loss, "vocabulary": train_data.vocab, "vocabulary_size": train_data.vocab_size, "hidden_size": N_HID}, "lstm_train_model.pth")    

    # Print loss every 10 epochs
    if i % 10 == 0:
        epoch_duration = time.time() - epoch_start
        print(f'Epoch {i}, training loss: {training_loss[-1]}, validation loss: {validation_loss[-1]}, duration: {epoch_duration:.2f}s', flush=True)

        
# Get first sentence in test set
inputs, targets, labels, lengths = test_data[0]

# One-hot encode input and target sequence
inputs_one_hot = one_hot_encode_sequence(inputs, test_data.vocab_size)

# Convert input to tensor
inputs_one_hot = torch.as_tensor(inputs_one_hot, dtype=torch.float32).permute(2,0,1)

# Convert target to tensor (not used currently, uncomment if used in the future)
# targets_ = torch.tensor(targets, dtype=tensor.long, device=device)

# Forward pass
net.eval()
with torch.no_grad():
    outputs = net(inputs_one_hot).cpu().numpy()

print('\nInput sequence:')
print(inputs)

print('\nTarget class:')
print(targets)

print('\nPredicted class:') 
print([idx_to_family[np.softmax(output)] for output in outputs])

# Plot training and validation loss
epoch = np.arange(len(training_loss))
plt.figure()
plt.plot(epoch, training_loss, 'r', label='Training loss',)
plt.plot(epoch, validation_loss, 'b', label='Validation loss')
plt.legend()
plt.xlabel('Epoch'), plt.ylabel('NLL')
plt.show()

torch.Size([129, 3])


ValueError: Expected input batch_size (129) to match target batch_size (0).